# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib widget

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

import panel as pn

import enderleaf.const as ec
from enderleaf.draw import image_grid, concat_tile_resize
from enderleaf.tools import read_dataframe, write_dataframe
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    ImageMergeMode,
    match_previous_rotation,
    get_circles
)
from enderleaf.draw import draw_circles

## Constants

In [ ]:
EXP = "Exp26DM05"
INOC = "I4"
PLATE = 1
DATE = "2026-05-15"

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
).dropna(subset="north")
df

## Select Cycle ID

In [ ]:
cycle_id = df.sample(n=1).iloc[0].cycle_id
cycle_id

In [ ]:
df_id = df[df.cycle_id==cycle_id]
df_id

In [ ]:
accu, cx, cy, r = get_circles(load(df_id.iloc[0]), color_space="hsv", channel="s")[
    "accepted"
][0]
crop_data = Rectangle.from_circle((cx, cy, r + 16))

to_pil(
    concat_tile_resize(
        [
            [
                crop_image(load(df_id.iloc[0]), crop_data),
                merge_images(
                    image_list=[
                        crop_image(load(row[1]), crop_data) for row in df_id.iterrows()
                    ],
                    merge_mode=ImageMergeMode.MIN,
                ),
            ]
        ]
    )
)